# About PyTorch

For these models' implementation PyTorch will be used alongside its common structure. For information on in-code structural decisions, check: 
https://docs.pytorch.org/docs/2.12/generated/torch.nn.RNN.html
https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html
https://docs.pytorch.org/docs/2.12/generated/torch.nn.Module.html

# Code
## Libraries

In [73]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [74]:
data = pd.read_csv("../data/Load/data_sensor_D.csv")
data.head()
 

,periodo,datetime,CO2_ppm,AQI,CO_ppm,SO2_ppm,O3_ppm,NO2_ppm,PM2_5_ugm3,temp_C,hum_pct,Bat_pct
0,P1,2025-11-06 20:09:35,0.997565,5,0.446300,0.930365,-0.340872,-0.584938,0.443770,0.340532,0.434159,0.704261
1,P1,2025-11-06 20:10:18,0.837793,5,0.358256,1.085778,-0.113082,-0.303829,0.380526,0.338870,0.435463,0.700501
2,P1,2025-11-06 20:11:19,0.651854,4,0.311917,1.085778,-0.204198,-0.491235,0.380526,0.337209,0.432855,0.701754
3,P1,2025-11-06 20:12:19,0.530929,4,0.324274,1.023613,-0.158640,-0.444384,0.317281,0.337209,0.438070,0.703008
4,P1,2025-11-06 20:13:20,0.449070,4,0.284113,1.054696,-0.249756,-0.444384,0.317281,0.332226,0.436767,0.704261


In [75]:

numeric_data = data.drop(columns=["periodo", "datetime"]).copy()
X = numeric_data.drop(columns=["AQI"]).to_numpy()
y = numeric_data["AQI"].to_numpy()
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [76]:
#Transform and Reshape data

scaler_X = StandardScaler()
scaler_y = StandardScaler()
 
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)
 
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_sc  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

In [77]:
def to_tensor_3d(X, y):
    """
    Function that turns data into tensors, such that it is compatible
    with PyTorch's RNN API
    """
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)   # (N, 1, F)
    y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)   # (N, 1)
    return TensorDataset(X_t, y_t)

train_ds = to_tensor_3d(X_train_sc, y_train_sc)
test_ds  = to_tensor_3d(X_test_sc,  y_test_sc)
 
BATCH_SIZE = 64


# Considering no sliding window, assuming it is not sequential
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
 

In [78]:
# Generalized, where to store computations? in case user has a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [79]:
# Summarized, to use for both models
INPUT_SIZE  = X_train.shape[1]
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
DROPOUT     = 0.2
EPOCHS      = 100
LR          = 1e-3

## Classes and functions
Generalized for use in other proyects

In [80]:
class VanillaRNN(nn.Module):
    """Single-step regression with a vanilla (Elman) RNN."""
 
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            nonlinearity="tanh",
        )
        self.fc = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        out, _ = self.rnn(x) # out: (batch, seq, hidden)
        return self.fc(out[:, -1, :]) # last time-step → (batch, 1)

In [81]:
class GRUNet(nn.Module):
    """Single-step regression with a GRU."""
 
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

In [82]:
def train_model(model, loader, epochs=EPOCHS, lr=LR):
    """
    Function that manages and tracks training stage's epochs
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )
 
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(X_batch)
 
        avg_loss = epoch_loss / len(loader.dataset)
        scheduler.step(avg_loss)
 
        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d}/{epochs}  loss={avg_loss:.6f}")
 
    return model

In [83]:
def evaluate_model(model, loader, scaler_y):
    """
    Function that predicts and perfurms evaluation metrics for models
    """
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            out = model(X_batch).cpu().numpy()
            preds.append(out)
            targets.append(y_batch.numpy())
 
    preds   = scaler_y.inverse_transform(np.vstack(preds)).ravel()
    targets = scaler_y.inverse_transform(np.vstack(targets)).ravel()
 
    rmse = np.sqrt(mean_squared_error(targets, preds))
    mae  = mean_absolute_error(targets, preds)
    return rmse, mae, preds, targets

In [84]:
# --------------------------------
#           Vanilla RNN
# --------------------------------


print("\n── Vanilla RNN ──────────────────────────────────────────────────────")
rnn_model = VanillaRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
rnn_model = train_model(rnn_model, train_loader)
 
rmse_rnn, mae_rnn, y_pred_rnn, _ = evaluate_model(rnn_model, test_loader, scaler_y)
print(f"Vanilla RNN RMSE: {rmse_rnn:.4f}")
print(f"Vanilla RNN MAE:  {mae_rnn:.4f}")
 

# --------------------------------
#               GRU
# --------------------------------
print("\n── GRU ──────────────────────────────────────────────────────────────")
gru_model = GRUNet(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
gru_model = train_model(gru_model, train_loader)
 
rmse_gru, mae_gru, y_pred_gru, y_true = evaluate_model(gru_model, test_loader, scaler_y)
print(f"GRU RMSE: {rmse_gru:.4f}")
print(f"GRU MAE:  {mae_gru:.4f}")
 



── Vanilla RNN ──────────────────────────────────────────────────────
  Epoch  10/100  loss=0.100537
  Epoch  20/100  loss=0.091852
  Epoch  30/100  loss=0.077795
  Epoch  40/100  loss=0.074134
  Epoch  50/100  loss=0.072588
  Epoch  60/100  loss=0.071133
  Epoch  70/100  loss=0.071233
  Epoch  80/100  loss=0.070322
  Epoch  90/100  loss=0.070135
  Epoch 100/100  loss=0.070109
Vanilla RNN RMSE: 0.2459
Vanilla RNN MAE:  0.1826

── GRU ──────────────────────────────────────────────────────────────
  Epoch  10/100  loss=0.098972
  Epoch  20/100  loss=0.095285
  Epoch  30/100  loss=0.094575
  Epoch  40/100  loss=0.092659
  Epoch  50/100  loss=0.090732
  Epoch  60/100  loss=0.090566
  Epoch  70/100  loss=0.090083
  Epoch  80/100  loss=0.090054
  Epoch  90/100  loss=0.090263
  Epoch 100/100  loss=0.089576
GRU RMSE: 0.2792
GRU MAE:  0.2255


In [85]:
# --------------------------------
#           Summary Metrics
# --------------------------------
print("\n── Results summary ──────────────────────────────────────────────────")
results = pd.DataFrame({
    "Model": ["Vanilla RNN", "GRU"],
    "RMSE":  [rmse_rnn,     rmse_gru],
    "MAE":   [mae_rnn,      mae_gru],
})
print(results.to_string(index=False))


── Results summary ──────────────────────────────────────────────────
      Model     RMSE      MAE
Vanilla RNN 0.245893 0.182631
        GRU 0.279245 0.225537


## Cross Validation GRU Model

GRU Hyperparameter Optimization via K-Fold Cross-Validation. Searches over hidden_size, num_layers, dropout, lr, batch_size using K-Fold CV, then retrains the best config on the full training set. Selected ranges are semi-arbitrary.

In [86]:
import itertools
from sklearn.model_selection import KFold

In [87]:
param_grid = {
    "hidden_size": [32, 64, 128],
    "num_layers":  [1, 2],
    "dropout":     [0.0, 0.2, 0.3],
    "lr":          [1e-3, 5e-4],
    "batch_size":  [32, 64],
}

CV_FOLDS    = 5
CV_EPOCHS   = 50   
RANDOM_SEED = 42


### Helpers in Generalized Form

In [88]:

def make_fold_loaders(X, y, train_idx, val_idx, batch_size):
    """
    Helper Function
    build fold loaders (scale inside fold to avoid leakage)
    idea suggested from Stack Overflow
    """
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    fold_scaler_X = StandardScaler()
    fold_scaler_y = StandardScaler()

    X_tr_sc  = fold_scaler_X.fit_transform(X_tr)
    X_val_sc = fold_scaler_X.transform(X_val)

    y_tr_sc  = fold_scaler_y.fit_transform(y_tr.reshape(-1, 1)).ravel()
    y_val_sc = fold_scaler_y.transform(y_val.reshape(-1, 1)).ravel()

    tr_loader  = DataLoader(to_tensor_3d(X_tr_sc,  y_tr_sc),
                             batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(to_tensor_3d(X_val_sc, y_val_sc),
                             batch_size=batch_size, shuffle=False)
    return tr_loader, val_loader, fold_scaler_y


def train_fold(model, loader, epochs, lr):
    """
    Helper
    single-fold training
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )
    for _ in range(epochs):
        model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(X_batch)
        scheduler.step(epoch_loss / len(loader.dataset))
    return model

def fold_evaluate(model, loader, fold_scaler_y):
    """
    HELPER 
    RMSE on a validation loader (original scale)
    """
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            preds.append(model(X_batch).cpu().numpy())
            targets.append(y_batch.numpy())
    preds   = fold_scaler_y.inverse_transform(np.vstack(preds)).ravel()
    targets = fold_scaler_y.inverse_transform(np.vstack(targets)).ravel()
    return np.sqrt(mean_squared_error(targets, preds))


### CV Applied

In [89]:

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values())) # taken from Claude
kf     = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

print(f"Total combinations : {len(combos)}")
print(f"Folds              : {CV_FOLDS}")
print(f"Total training runs: {len(combos) * CV_FOLDS}\n")


Total combinations : 72
Folds              : 5
Total training runs: 360



In [90]:

cv_results = []

for i, combo in enumerate(combos):
    params     = dict(zip(keys, combo))
    fold_rmses = []

    for tr_idx, val_idx in kf.split(X_train):
        tr_loader, val_loader, fold_sy = make_fold_loaders(
            X_train, y_train, tr_idx, val_idx, params["batch_size"]
        )
        model = GRUNet(
            input_size  = INPUT_SIZE,
            hidden_size = params["hidden_size"],
            num_layers  = params["num_layers"],
            dropout     = params["dropout"],
        )
        model = train_fold(model, tr_loader, CV_EPOCHS, params["lr"])
        fold_rmses.append(fold_evaluate(model, val_loader, fold_sy))

    mean_rmse = float(np.mean(fold_rmses))
    std_rmse  = float(np.std(fold_rmses))
    cv_results.append({**params, "mean_rmse": mean_rmse, "std_rmse": std_rmse})

    print(f"[{i+1:3d}/{len(combos)}] {params}  →  "
          f"RMSE = {mean_rmse:.4f} ± {std_rmse:.4f}")


[  1/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.001, 'batch_size': 32}  →  RMSE = 0.2755 ± 0.0010
[  2/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.001, 'batch_size': 64}  →  RMSE = 0.2764 ± 0.0006
[  3/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.0005, 'batch_size': 32}  →  RMSE = 0.2778 ± 0.0015
[  4/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.0005, 'batch_size': 64}  →  RMSE = 0.2796 ± 0.0008
[  5/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32}  →  RMSE = 0.2749 ± 0.0005
[  6/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 64}  →  RMSE = 0.2766 ± 0.0017
[  7/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.2, 'lr': 0.0005, 'batch_size': 32}  →  RMSE = 0.2771 ± 0.0007
[  8/72] {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.2, 'lr': 0.0005, 'batch_size': 64}  →  RMSE = 0.2786 ± 0.0005
[  9/72] {'hidden_size': 32, 'num_la

### Results

In [91]:

results_df = (pd.DataFrame(cv_results)
                .sort_values("mean_rmse")
                .reset_index(drop=True))

print(results_df.head(3).to_string(index=False))


best = results_df.iloc[0]
print("\n ***** Best hyperparameters **************************")
print(best.to_string())


 hidden_size  num_layers  dropout    lr  batch_size  mean_rmse  std_rmse
         128           2      0.2 0.001          32   0.196245  0.010818
         128           2      0.0 0.001          32   0.200841  0.008567
         128           2      0.3 0.001          32   0.217057  0.030407

 ***** Best hyperparameters **************************
hidden_size    128.000000
num_layers       2.000000
dropout          0.200000
lr               0.001000
batch_size      32.000000
mean_rmse        0.196245
std_rmse         0.010818


## Train GRU with best hyperparameters

In [92]:
best_gru = GRUNet(
    input_size  = INPUT_SIZE,
    hidden_size = int(best["hidden_size"]),
    num_layers  = int(best["num_layers"]),
    dropout     = float(best["dropout"]),
)

best_train_loader = DataLoader(
    train_ds, batch_size=int(best["batch_size"]), shuffle=True
)

print("\nRetraining best GRU on the full training set …")
best_gru = train_model(best_gru, best_train_loader,
                        epochs=EPOCHS, lr=float(best["lr"]))

rmse_best, mae_best, _, _ = evaluate_model(best_gru, test_loader, scaler_y)
print(f"\nBest GRU (CV-tuned)  RMSE : {rmse_best:.4f}")
print(f"Best GRU (CV-tuned)  MAE  : {mae_best:.4f}")


Retraining best GRU on the full training set …
  Epoch  10/100  loss=0.096658
  Epoch  20/100  loss=0.092530
  Epoch  30/100  loss=0.057348
  Epoch  40/100  loss=0.044024
  Epoch  50/100  loss=0.042238
  Epoch  60/100  loss=0.040936
  Epoch  70/100  loss=0.039964
  Epoch  80/100  loss=0.039101
  Epoch  90/100  loss=0.036378
  Epoch 100/100  loss=0.035725

Best GRU (CV-tuned)  RMSE : 0.1838
Best GRU (CV-tuned)  MAE  : 0.0723


In [93]:
# Compare with baseline GRU
print(f"\nBaseline GRU         RMSE : {rmse_gru:.4f}")
print(f"Baseline GRU         MAE  : {mae_gru:.4f}")


Baseline GRU         RMSE : 0.2792
Baseline GRU         MAE  : 0.2255
